#1. Basic Tasks 

##1. Use MERGE INTO to upsert a batch of 'changed customer' records into a silver customers table. 

In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_raw (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_raw
VALUES
  (1, 'John Smith',     'john.smith@email.com',     '555-0101', 'New York',    'NY', 'Retail',     true),
  (2, 'Jane Doe',       'jane.doe@email.com',       '555-0102', 'Los Angeles', 'CA', 'Wholesale',  true),
  (3, 'Bob Johnson',    'bob.johnson@email.com',    '555-0103', 'Chicago',     'IL', 'Retail',     true),
  (4, 'Alice Brown',    'alice.brown@email.com',    '555-0104', 'Houston',     'TX', 'Enterprise', true),
  (5, 'Charlie Wilson', 'charlie.wilson@email.com', '555-0105', 'Phoenix',     'AZ', 'Retail',     true);


CREATE OR REPLACE TABLE cyntexa_dev.silver.customers AS
SELECT
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  current_timestamp() AS last_updated
FROM cyntexa_dev.bronze.customers_raw;

SELECT * FROM cyntexa_dev.silver.customers ORDER BY customer_id;


CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_staging (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_staging
VALUES
  (1, 'John Smith',     'john.smith@new-email.com',  '555-0101', 'Boston',  'MA', 'Enterprise', true),
  (3, 'Bob Johnson',    'bob.johnson@email.com',    '555-9999', 'Chicago', 'IL', 'Wholesale', true),   
  (5, 'Charlie Wilson', 'charlie.wilson@email.com', '555-0105', 'Phoenix', 'AZ', 'Retail',    false),  
  (6, 'Diana Prince',   'diana.prince@email.com',   '555-0106', 'Seattle', 'WA', 'Enterprise', true),
  (7, 'Edward Norton',  'edward.norton@email.com',  '555-0107', 'Denver',  'CO', 'Retail',     true);

MERGE WITH SCHEMA EVOLUTION INTO cyntexa_dev.silver.customers AS target
USING cyntexa_dev.bronze.customers_staging AS source
ON target.customer_id = source.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
  

SELECT * FROM cyntexa_dev.silver.customers ORDER BY customer_id;

##2. Grant SELECT on a table to one group and a masked/limited view to another group using Unity Catalog permissions. 



###Permission Management

#### Create User Groups:
```sql
-- Run these in Databricks Account Console or via API
-- Groups: data_engineers, analysts, executives, sales_us, sales_eu
```

#### Grant Permissions:
```sql
-- Schema-level permissions
GRANT USAGE ON SCHEMA dev.bronze TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.silver TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `data_engineers`;
GRANT USAGE ON SCHEMA dev.gold TO `analysts`;  -- Read-only for analysts

-- Table-level permissions
GRANT SELECT ON TABLE dev.gold.customer_metrics TO `analysts`;
GRANT SELECT ON TABLE dev.gold.sales_summary TO `analysts`;
GRANT ALL PRIVILEGES ON TABLE dev.bronze.customers_raw TO `data_engineers`;

-- Secured view access
GRANT SELECT ON VIEW dev.gold.customers_masked TO `analysts`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_us`;
GRANT SELECT ON VIEW dev.gold.sales_secure TO `sales_eu`;

-- Revoke direct access to sensitive tables
REVOKE SELECT ON TABLE dev.silver.customers_clean FROM `analysts`;
```

##3. Look up the DBU consumption for a compute resource you've been using and explain, in plain terms, what a DBU is billing for. 

In [0]:
SELECT 
  account_id,
  usage_date,
  sku_name,
  cloud,
  usage_unit,
  usage_quantity,
  usage_metadata.*
FROM system.billing.usage
WHERE usage_unit = 'DBU'
ORDER BY usage_date DESC, usage_quantity DESC

### What is a DBU (Databricks Unit)?

A **DBU** is Databricks' unit of processing capability. In plain terms:

- **What you're paying for**: The compute power (CPU, memory, and resources) that Databricks provisions to run your workloads
- **How it works**: Different workload types consume DBUs at different rates:
  - **Jobs Compute**: Running scheduled ETL jobs and batch processing
  - **All-Purpose Compute**: Interactive notebook development and ad-hoc queries  
  - **SQL Warehouses**: Running SQL queries and dashboards

Think of it like electricity: a DBU measures the "processing power" you use, similar to how kilowatt-hours measure electrical power consumption.

##4. Build a full SCD Type 2 table: implement the MERGE that closes out old records (setting end_date and is_current) and inserts new versions when a tracked column changes. 

In [0]:

CREATE OR REPLACE TABLE cyntexa_dev.silver.customers_scd2 (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN,
  start_date     TIMESTAMP,
  end_date       TIMESTAMP
);


INSERT INTO cyntexa_dev.silver.customers_scd2
SELECT 
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  current_timestamp() AS start_date,
  NULL AS end_date
FROM cyntexa_dev.bronze.customers_raw;

SELECT * FROM cyntexa_dev.silver.customers_scd2 ORDER BY customer_id, start_date;


CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_scd2_staging (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
);

INSERT INTO cyntexa_dev.bronze.customers_scd2_staging
VALUES
  (1, 'John Smith',     'john.smith@updated.com',  '555-1111', 'Boston',    'MA', 'Enterprise', true),  
  (2, 'Jane Doe',       'jane.doe@email.com',      '555-0102', 'Los Angeles', 'CA', 'Wholesale',  true), 
  (3, 'Bob Johnson',    'bob.johnson@email.com',   '555-9999', 'Chicago',   'IL', 'Wholesale',  false), 
  (6, 'Diana Prince',   'diana.prince@email.com',  '555-0106', 'Seattle',   'WA', 'Enterprise', true);  

MERGE INTO cyntexa_dev.silver.customers_scd2 AS target
USING (
  SELECT 
    s.*,
    current_timestamp() AS merge_timestamp
  FROM cyntexa_dev.bronze.customers_scd2_staging s
) AS source
ON target.customer_id = source.customer_id 
   AND target.is_active = true

WHEN MATCHED AND (
  target.customer_name != source.customer_name OR
  target.email != source.email OR
  target.phone != source.phone OR
  target.city != source.city OR
  target.state != source.state OR
  target.segment != source.segment OR
  target.is_active != source.is_active
)
THEN UPDATE SET
  target.end_date = source.merge_timestamp,
  target.is_active = false

WHEN NOT MATCHED THEN INSERT (
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  start_date,
  end_date
) VALUES (
  source.customer_id,
  source.customer_name,
  source.email,
  source.phone,
  source.city,
  source.state,
  source.segment,
  source.is_active,
  source.merge_timestamp,
  NULL
);


INSERT INTO cyntexa_dev.silver.customers_scd2
SELECT 
  s.customer_id,
  s.customer_name,
  s.email,
  s.phone,
  s.city,
  s.state,
  s.segment,
  s.is_active,
  current_timestamp() AS start_date,
  NULL AS end_date
FROM cyntexa_dev.bronze.customers_scd2_staging s
INNER JOIN (
  SELECT customer_id
  FROM cyntexa_dev.silver.customers_scd2
  WHERE is_active = false
    AND end_date IS NOT NULL
    AND end_date >= date_sub(current_timestamp(), 1)  -- Changed in this run
  GROUP BY customer_id
) closed ON s.customer_id = closed.customer_id;

-- Step 6: View the results showing full history
SELECT 
  customer_id,
  customer_name,
  email,
  city,
  state,
  segment,
  start_date,
  end_date
FROM cyntexa_dev.silver.customers_scd2
ORDER BY customer_id, start_date;

##5. Query the SCD Type 2 table to answer a point-in-time question, e.g. 'what was this customer's address as of March 1st?'

In [0]:
-- Point-in-time query: What was customer #1's address as of a specific date?
SELECT 
  customer_id,
  customer_name,
  email,
  city,
  state,
  segment,
  is_active,
  start_date,
  end_date
FROM cyntexa_dev.silver.customers_scd2
WHERE customer_id = 1
  AND start_date <= '2026-10-01 00:00:00'
  AND (end_date IS NULL OR end_date > '2026-09-01 00:00:00');

-- Query for all customers' addresses as of a specific point in time
SELECT 
  customer_id,
  customer_name,
  city,
  state,
  segment,
  is_active
FROM cyntexa_dev.silver.customers_scd2
WHERE start_date <= '2026-10-01 00:00:00'
  AND (end_date IS NULL OR end_date > '2026-09-01 00:00:00')
ORDER BY customer_id;

##6. Compare the estimated DBU cost of running a job on all-purpose vs. job compute, and recommend which Cyntexa should use for its nightly pipeline.


### DBU Cost Comparison: All-Purpose vs. Jobs Compute

#### Cost Analysis

**All-Purpose Compute:**
- **Use case**: Interactive development, ad-hoc queries, exploration
- **DBU rate**: ~2-3x more expensive than Jobs Compute
- **Billing**: Charged per minute of uptime (even when idle)
- **Typical pricing**: $0.40-0.75 per DBU (varies by cloud/region)

**Jobs Compute:**
- **Use case**: Scheduled/automated production workloads
- **DBU rate**: ~60-70% cheaper than All-Purpose Compute
- **Billing**: Charged only when actively running jobs
- **Typical pricing**: $0.07-0.15 per DBU (varies by cloud/region)
- **Auto-termination**: Automatically shuts down when job completes

#### Example Cost Calculation

For a nightly pipeline running 2 hours on a Medium cluster (~8 DBUs/hour):

```
All-Purpose Compute:
2 hours × 8 DBUs/hour × $0.55/DBU = $8.80 per night
× 30 days = $264/month

Jobs Compute:
2 hours × 8 DBUs/hour × $0.10/DBU = $1.60 per night
× 30 days = $48/month

Savings: $216/month (82% reduction)
```

#### Recommendation for Cyntexa's Nightly Pipeline

**Use Jobs Compute** for the following reasons:

1. **Cost Efficiency**: 60-80% lower DBU rates compared to All-Purpose
2. **Predictable Workload**: Nightly pipelines are scheduled, automated tasks—exactly what Jobs Compute is designed for
3. **No Idle Time**: Jobs Compute terminates immediately after completion, eliminating idle compute charges

**When to use All-Purpose Compute:**
- Interactive notebook development and testing
- Ad-hoc data exploration
- Prototyping new pipelines
- Debugging issues interactively



#3. Advanced Tasks

##7. Design a governance model for Cyntexa: which columns across which tables are sensitive (PII), which Unity Catalog groups should have access, and how you'd audit access


## Governance Model for Cyntexa

### 1. Sensitive Data Classification (PII Identification)

#### **Bronze Layer** (`cyntexa_dev.bronze`)

**Table: `customers_raw`**
- **Direct PII:**
  - `customer_name` - Full name
  - `email` - Email address
  - `phone` - Phone number
  - `city`, `state` - Geographic identifiers (quasi-identifiers when combined)
- **Non-Sensitive:**
  - `customer_id` - Business identifier
  - `segment` - Business classification
  - `is_active` - Status flag

#### **Silver Layer** (`cyntexa_dev.silver`)

**Table: `customers`**
- Same PII columns as bronze layer
- Additional metadata: `last_updated` (non-sensitive)

**Table: `customers_scd2`**
- Same PII columns as `customers`
- Historical tracking columns: `start_date`, `end_date`, `is_current` (non-sensitive)

---

### 2. Unity Catalog Group Access Model

#### **Recommended Groups:**

1. **`data_engineers`**
   - **Access:** Full read/write to bronze and silver layers
   - **PII Access:** Unrestricted (with audit trail)

2. **`data_analysts`**
   - **Access:** Read-only to masked/aggregated views in gold layer
   - **PII Access:** Masked or aggregated only

3. **`data_scientists`**
   - **Access:** Read access to silver layer with column-level masking
   - **PII Access:** Tokenized/hashed identifiers, masked contact info
---

### 3. Recommended Access Control Implementation

#### **Step 1: Create Masked Views for Analysts**

```sql
-- Gold layer: Masked customer view for analysts
CREATE OR REPLACE VIEW cyntexa_dev.gold.customers_masked AS
SELECT 
  customer_id,
  sha2(customer_name, 256) AS customer_name_hash,
  CONCAT(SUBSTRING(email, 1, 3), '***@', SPLIT(email, '@')[1]) AS email_masked,
  CONCAT(SUBSTRING(phone, 1, 3), '-****') AS phone_masked,
  state,  -- Keep state, mask city
  segment,
  is_active
FROM cyntexa_dev.silver.customers
WHERE is_current = true;
```

#### **Step 2: Apply Row Filters (ABAC Policies)**

```sql
-- Example: Region-based row filtering
CREATE FUNCTION cyntexa_dev.gold.customer_region_filter(state STRING)
RETURN 
  CASE 
    WHEN IS_MEMBER('regional_manager_west') THEN state IN ('CA', 'WA', 'AZ')
    WHEN IS_MEMBER('regional_manager_east') THEN state IN ('NY', 'MA')
    WHEN IS_MEMBER('data_engineers') THEN TRUE
    ELSE FALSE
  END;

ALTER TABLE cyntexa_dev.silver.customers 
SET ROW FILTER cyntexa_dev.gold.customer_region_filter ON (state);
```

#### **Step 3: Apply Column Masks**

```sql
-- Mask email for non-compliance users
CREATE FUNCTION cyntexa_dev.gold.mask_email(email STRING)
RETURN 
  CASE 
    WHEN IS_MEMBER('compliance_team') OR IS_MEMBER('data_engineers') THEN email
    ELSE CONCAT('***@', SPLIT(email, '@')[1])
  END;

ALTER TABLE cyntexa_dev.silver.customers 
ALTER COLUMN email SET MASK cyntexa_dev.gold.mask_email;
```

#### **Step 4: Grant Permissions**

```sql
-- Schema usage
GRANT USAGE ON SCHEMA cyntexa_dev.bronze TO `data_engineers`;
GRANT USAGE ON SCHEMA cyntexa_dev.silver TO `data_engineers`;
GRANT USAGE ON SCHEMA cyntexa_dev.gold TO `data_analysts`;
GRANT USAGE ON SCHEMA cyntexa_dev.gold TO `business_users`;

-- Data engineers: Full access to bronze and silver
GRANT ALL PRIVILEGES ON TABLE cyntexa_dev.bronze.customers_raw TO `data_engineers`;
GRANT ALL PRIVILEGES ON TABLE cyntexa_dev.silver.customers TO `data_engineers`;
GRANT ALL PRIVILEGES ON TABLE cyntexa_dev.silver.customers_scd2 TO `data_engineers`;

-- Analysts: Masked views only
GRANT SELECT ON VIEW cyntexa_dev.gold.customers_masked TO `data_analysts`;

-- Business users: Aggregated views only
GRANT SELECT ON VIEW cyntexa_dev.gold.customer_metrics TO `business_users`;

-- Compliance: Read-only to all layers
GRANT SELECT ON SCHEMA cyntexa_dev.silver TO `compliance_team`;
GRANT SELECT ON SCHEMA cyntexa_dev.bronze TO `compliance_team`;

-- Revoke direct access to PII tables from analysts
REVOKE SELECT ON TABLE cyntexa_dev.silver.customers FROM `data_analysts`;
REVOKE SELECT ON TABLE cyntexa_dev.bronze.customers_raw FROM `data_analysts`;
```

---

### 4. Audit Access Strategy

#### **A. Enable Unity Catalog Audit Logs**

```sql
-- Query audit logs for PII table access
SELECT 
  event_time,
  user_identity.email AS user_email,
  request_params.full_name_arg AS table_accessed,
  action_name,
  request_params.columns_accessed,
  source_ip_address
FROM system.access.audit
WHERE 
  action_name IN ('getTable', 'commandSubmit', 'readFiles')
  AND request_params.full_name_arg LIKE '%customers%'
  AND event_date >= current_date() - INTERVAL 7 DAYS
ORDER BY event_time DESC;
```

#### **B. Monitor Specific PII Column Access**

```sql
-- Track who accessed sensitive columns
SELECT 
  event_date,
  user_identity.email,
  request_params.full_name_arg AS table_name,
  request_params.command_text,
  CASE 
    WHEN request_params.command_text LIKE '%email%' OR 
         request_params.command_text LIKE '%phone%' OR 
         request_params.command_text LIKE '%customer_name%' 
    THEN 'PII_ACCESSED'
    ELSE 'NON_PII'
  END AS access_type
FROM system.access.audit
WHERE 
  action_name = 'commandSubmit'
  AND request_params.full_name_arg IN (
    'cyntexa_dev.bronze.customers_raw',
    'cyntexa_dev.silver.customers',
    'cyntexa_dev.silver.customers_scd2'
  )
  AND event_date >= current_date() - INTERVAL 30 DAYS
ORDER BY event_date DESC;
```

#### **C. Create Audit Dashboard**

```sql
-- Daily PII access summary
CREATE OR REPLACE VIEW cyntexa_dev.gold.pii_access_audit AS
SELECT 
  DATE(event_time) AS access_date,
  user_identity.email AS user_email,
  request_params.full_name_arg AS table_accessed,
  COUNT(*) AS access_count,
  MAX(event_time) AS last_access_time
FROM system.access.audit
WHERE 
  action_name IN ('getTable', 'commandSubmit')
  AND (
    request_params.full_name_arg LIKE '%customers%' OR
    request_params.command_text LIKE '%email%' OR
    request_params.command_text LIKE '%phone%' OR
    request_params.command_text LIKE '%customer_name%'
  )
  AND event_date >= current_date() - INTERVAL 90 DAYS
GROUP BY 
  DATE(event_time),
  user_identity.email,
  request_params.full_name_arg
ORDER BY access_date DESC, access_count DESC;

-- Grant to compliance team
GRANT SELECT ON VIEW cyntexa_dev.gold.pii_access_audit TO `compliance_team`;
```

#### **D. Set Up Alerting**

```sql
-- Identify unusual access patterns
SELECT 
  user_identity.email,
  COUNT(DISTINCT request_params.full_name_arg) AS tables_accessed,
  COUNT(*) AS total_queries,
  COLLECT_SET(request_params.full_name_arg) AS tables_list
FROM system.access.audit
WHERE 
  action_name = 'commandSubmit'
  AND request_params.full_name_arg LIKE '%customers%'
  AND event_date = current_date()
GROUP BY user_identity.email
HAVING COUNT(*) > 100  -- Flag users with >100 queries per day
ORDER BY total_queries DESC;
```

---

### 5. Governance Best Practices Summary

#### **Data Classification**
- **Tier 1 (Highly Sensitive):** `customer_name`, `email`, `phone`
- **Tier 2 (Quasi-Identifiers):** `city`, `state`, `customer_id` (when combined)
- **Tier 3 (Non-Sensitive):** `segment`, `is_active`, `last_updated`


#### **Regular Reviews**
- **Quarterly:** Review group memberships and access grants
- **Monthly:** Analyze audit logs for anomalous access patterns
- **On-Demand:** Investigate alerts for policy violations

##8. Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the edge case of a customer record that hasn't changed since the last load (it should not create a false new version).

In [0]:
-- Extended SCD Type 2: Track changes across multiple columns simultaneously
CREATE OR REPLACE TABLE cyntexa_dev.bronze.customers_extended_staging (
  customer_id    INT,
  customer_name  STRING,
  email          STRING,
  phone          STRING,
  city           STRING,
  state          STRING,
  segment        STRING,
  is_active      BOOLEAN
  );

-- Step 2: Insert data
INSERT INTO cyntexa_dev.bronze.customers_extended_staging
VALUES
  (1, 'John Smith',     'john.smith@updated.com',  '555-1111', 'Boston',      'MA', 'Enterprise', true),  -- Changed: email, city, state, segment
  (2, 'Jane Doe',       'jane.doe@email.com',      '555-0102', 'Los Angeles', 'CA', 'Wholesale',  true),  -- NO CHANGE
  (3, 'Bob Johnson',    'bob.johnson@email.com',   '555-9999', 'Chicago',     'IL', 'Wholesale',  false), -- Changed: phone, segment, is_active
  (6, 'Diana Prince',   'diana.prince@email.com',  '555-0106', 'Seattle',     'WA', 'Enterprise', true),  -- NEW RECORD
  (8, 'Frank Miller',   'frank.miller@email.com',  '555-0108', 'Austin',      'TX', 'Retail',     true);  -- NEW RECORD

-- Step 3: Enhanced MERGE that only creates new versions when data actually changed
MERGE INTO cyntexa_dev.silver.customers_scd2 AS target
USING (
  SELECT 
    s.*,
    current_timestamp() AS merge_timestamp
  FROM cyntexa_dev.bronze.customers_extended_staging s
) AS source
ON target.customer_id = source.customer_id 
   AND target.is_active = true

-- Only update (close out) records where at least one tracked column has changed
WHEN MATCHED AND (
  -- Compare ALL tracked columns to detect changes
  target.customer_name != source.customer_name OR
  target.email != source.email OR
  target.phone != source.phone OR
  target.city != source.city
)
THEN UPDATE SET
  target.end_date = source.merge_timestamp,
  target.is_active = false

-- Insert new records that don't exist yet
WHEN NOT MATCHED THEN INSERT (
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  is_active,
  start_date,
  end_date
) VALUES (
  source.customer_id,
  source.customer_name,
  source.email,
  source.phone,
  source.city,
  source.state,
  source.segment,
  source.is_active,
  source.merge_timestamp,
  NULL
);

-- Step 4: Insert new versions for records that were closed out in Step 3
INSERT INTO cyntexa_dev.silver.customers_scd2
SELECT 
  s.customer_id,
  s.customer_name,
  s.email,
  s.phone,
  s.city,
  s.state,
  s.segment,
  s.is_active,
  current_timestamp() AS start_date,
  NULL AS end_date
FROM cyntexa_dev.bronze.customers_extended_staging s
INNER JOIN (
  -- Find records that were just closed (changed records)
  SELECT customer_id
  FROM cyntexa_dev.silver.customers_scd2
  WHERE is_active = false
    AND end_date IS NOT NULL
    AND end_date >= date_sub(current_timestamp(), 1)  -- Changed in this run
  GROUP BY customer_id
) closed ON s.customer_id = closed.customer_id;

-- Step 5: Verify results
-- Show full history with all tracked columns
SELECT 
  customer_id,
  customer_name,
  email,
  phone,
  city,
  state,
  segment,
  start_date,
  end_date,
  is_active
FROM cyntexa_dev.silver.customers_scd2
ORDER BY customer_id, start_date;

##9. (Data Analyst) Using the SCD Type 2 history table, build a customer retention/churn-over-time report that depends on point-in-time correctness, and explain why a simple 'current state' table would give the wrong answer here.

In [0]:
-- Customer Retention/Churn-Over-Time Report using SCD Type 2

WITH monthly_snapshots AS (
  SELECT 
    DATE_TRUNC('MONTH', snapshot_date) AS month,
    customer_id,
    customer_name,
    segment,
    city,
    state
  FROM (
    SELECT 
      ADD_MONTHS(LAST_DAY(CURRENT_DATE()), -seq.month_offset) AS snapshot_date
    FROM (
      SELECT EXPLODE(SEQUENCE(0, 11)) AS month_offset
    ) seq
  ) dates
  CROSS JOIN cyntexa_dev.silver.customers_scd2 c
  WHERE 
    -- Point-in-time logic: customer record was active at month-end
    c.start_date <= dates.snapshot_date
    AND (c.end_date IS NULL OR c.end_date > dates.snapshot_date)
),

-- Step 2: Identify customer status changes month-over-month
customer_status AS (
  SELECT 
    current.month,
    current.customer_id,
    current.segment,
    CASE 
      WHEN previous.customer_id IS NULL THEN 'New'
      WHEN current.customer_id IS NOT NULL AND previous.customer_id IS NOT NULL THEN 'Retained'
      ELSE 'Other'
    END AS status
  FROM monthly_snapshots current
  LEFT JOIN monthly_snapshots previous 
    ON current.customer_id = previous.customer_id
    AND current.month = ADD_MONTHS(previous.month, 1)
),

-- Step 3: Identify churned customers (present last month, gone this month)
churned_customers AS (
  SELECT 
    ADD_MONTHS(previous.month, 1) AS month,
    previous.customer_id,
    previous.segment,
    'Churned' AS status
  FROM monthly_snapshots previous
  LEFT JOIN monthly_snapshots current
    ON previous.customer_id = current.customer_id
    AND ADD_MONTHS(previous.month, 1) = current.month
  WHERE current.customer_id IS NULL
),

-- Step 4: Combine all customer statuses
all_statuses AS (
  SELECT * FROM customer_status
  UNION ALL
  SELECT * FROM churned_customers
)

-- Step 5: Generate the retention/churn report by month and segment
SELECT 
  month,
  segment,
  COUNT(CASE WHEN status = 'New' THEN 1 END) AS new_customers,
  COUNT(CASE WHEN status = 'Retained' THEN 1 END) AS retained_customers,
  COUNT(CASE WHEN status = 'Churned' THEN 1 END) AS churned_customers,
  COUNT(CASE WHEN status IN ('New', 'Retained') THEN 1 END) AS total_active_customers,
  
  -- Retention rate: % of last month's customers still active this month
  ROUND(
    COUNT(CASE WHEN status = 'Retained' THEN 1 END) * 100.0 / 
    NULLIF(COUNT(CASE WHEN status = 'Retained' THEN 1 END) + 
           COUNT(CASE WHEN status = 'Churned' THEN 1 END), 0),
    2
  ) AS retention_rate_pct,
  
  -- Churn rate: % of last month's customers who churned this month
  ROUND(
    COUNT(CASE WHEN status = 'Churned' THEN 1 END) * 100.0 / 
    NULLIF(COUNT(CASE WHEN status = 'Retained' THEN 1 END) + 
           COUNT(CASE WHEN status = 'Churned' THEN 1 END), 0),
    2
  ) AS churn_rate_pct
FROM all_statuses
GROUP BY month, segment
ORDER BY month DESC, segment;





### Why a Simple 'Current State' Table Would Give the Wrong Answer

The retention and churn analysis above relies on **point-in-time correctness** — the ability to answer "which customers were active on any past date?" A simple "current state" table (showing only the latest version of each customer record) would produce **incorrect results** for several critical reasons:

---

#### **1. Lost Historical Context**

A current-state table only shows who is active **today**. It cannot tell you:
- Which customers were active 6 months ago
- When a customer churned (the exact date they became inactive)
- Whether a customer was retained month-over-month in the past

---

#### **2. Survivor Bias in Retention Metrics**

Retention rate requires comparing:
- Customers active in **Period T** (e.g., January)
- How many of those **same customers** are still active in **Period T+1** (e.g., February)

A current-state table only contains customers who:
- Are currently active, OR
- Became inactive recently (if you keep one "inactive" record per customer)

You **cannot reconstruct** which customers existed in prior months because:
- Churned customers may have been deleted or overwritten
- You don't know **when** they became inactive relative to past reporting periods

---

#### **3. Inability to Track Churn Timing**

Churn rate = (Customers who left during period T) / (Customers active at start of T)

A current-state table shows customers as either "active" or "inactive" **today**, but:
- Without `start_date` and `end_date` (from SCD Type 2), you cannot assign churn events to the correct month


---

#### **4. Retroactive Changes Are Invisible**

SCD Type 2 tracks **all versions** of a customer record:
- If a customer's segment changed from "Retail" to "Enterprise" in March
- The SCD Type 2 table preserves both versions with their validity periods
- A January retention report should use the **January version** of the segment ("Retail")

A current-state table only shows the **latest** segment ("Enterprise"):
- January reports would incorrectly attribute the customer to "Enterprise" retention
- This **rewrites history** — the customer wasn't Enterprise in January

---

**Conclusion:** Retention and churn are metrics that require knowing the state of each customer **at multiple points in time**. A current-state table destroys this history, making accurate retention/churn analysis impossible. SCD Type 2's `start_date` and `end_date` preserve the full timeline, enabling correct point-in-time queries for any historical period.
